[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/fluids/sphere_vortices/sphere_vortices.ipynb)

# Vortices on the Sphere

This example moves point vortices over a sphere by the flow they make together. A point vortex is a point with a circulation around it: the fluid swirls around it and is irrotational everywhere else. Equal vortices in a ring turn rigidly about its centre, and on the sphere a ring of seven, on the edge of stability in the plane, breaks up; a pair of opposite vortices travels around the whole sphere, and two such pairs leapfrog as they go.

A vortex is modelled as a bivector of the geometric algebra of the sphere, the meet of two great circles oriented by the circulation, standing for the antipodal pair of points its rotor leaves fixed: the flow is symmetric between them, its vorticity opposite at the two. The flow at a place is the sum of the vortices' bivectors, each weighted by its rate there, one bivector that generates the turn the fluid makes, and the rate is the expression of the plane in the join `A & B` of vortex and place, whose norm is a distance there and a sine here; only the algebra's signature differs. Two time steps are built from rotors: one turns pairs of vortices by their exact rotors, the other turns every vortex by the rotor of the flow at the middle of its step.

In the notation of point-vortex dynamics on the projective plane, where a vortex on the sphere comes with its antipodal image of opposite circulation, the velocity reads $\dot{\mathbf x}_k = \frac{1}{2\pi} \sum_{j \ne k} \Gamma_j \frac{\mathbf x_j \times \mathbf x_k}{|\mathbf x_j \times \mathbf x_k|^2}$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from dataclasses import dataclass, replace

import jax
import numpy as np

from numga import NumpyContext, concatenate
from numga.algebras import Spherical3D as ga
from numga.backend.jax import JaxContext
from examples.util.animation import show_animation
from examples.fluids.sphere_vortices import render
from examples.fluids.sphere_vortices.core import schedule

In [ ]:
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                # Scalar+
Pivot = ga.gatype.bivector()

## 1. Vortices as bivectors

A vortex is a bivector `A`: a point of the sphere, the meet of two great circles, oriented by the circulation around it, standing for the vortex's two antipodal points. A place on the sphere is held the same way, by its bivector `B`. The rotor of a vortex of circulation $\Gamma$ turns every place about the vortex, at the rate $\Gamma / 2\pi$ over the squared norm of the join `A & B`, the great circle through the vortex and the place, whose norm is the sine of the angle between them, the same from either of the vortex's two points. Along each circle about the vortex the speed is $\Gamma / 2\pi$ over the circle's radius, so the circulation around every such circle is $\Gamma$, and the flow is irrotational everywhere except at the two points, where its vorticity sits, $\Gamma$ at the one and $-\Gamma$ at the other, as on a closed surface it must. The rate is the expression of the plane, where the norm of the join of two points is their distance; only the algebra's signature differs. A core radius softens it near the points.

A vortex with both its bivector and its circulation negated turns the sphere the same way. The flow at a place is the sum of the vortices' bivectors, each weighted by its rate there, one bivector that generates the turn the fluid makes there, and a vortex moves with the flow of the others.

In [ ]:
# The angles from the pole along a meridian of the places the flow is measured at.
angles = np.array([0.2, 1.0, np.pi / 2, 2.6])

In [ ]:
@dataclass(frozen=True)
class Vortices:
    pivots: Pivot                                                                    # [..., vortices] Pivot+
    # The circulation of each vortex.
    circulations: np.ndarray                                                         # [vortices]
    core_radius: float


def pivot(directions: np.ndarray) -> Pivot:
    """The pivots through the centre along the given directions [..., 3], of unit size: on the bivectors yz, zx and xy, the meets of planes that are the x, y and z pivots."""
    return mv.bivector(directions / np.linalg.norm(directions, axis=-1, keepdims=True))


def flow(vortices: Vortices, at: Pivot) -> Pivot:
    """The flow of the other vortices at each vortex's place: their bivectors, each weighted by its rate there."""
    squared = (at.normalized()[..., :, None] & vortices.pivots.normalized()[..., None, :]).norm_squared()   # [..., at, vortices] Scalar+
    others = vortices.circulations * ~np.eye(len(vortices.circulations), dtype=bool)   # [at, vortices]
    rates = others / (2 * np.pi * (squared + vortices.core_radius**2))               # [..., at, vortices] Scalar+
    return (vortices.pivots.normalized()[..., None, :] * rates).sum(axis=-1)         # [..., at] Pivot+


def turned(colatitudes: np.ndarray, longitudes: np.ndarray) -> Pivot:
    """Points at the given angles from the pole and around it: the pole turned about the point (0, 1, 0), then about itself."""
    across = pivot(np.array([0.0, 1.0, 0.0]))                                        # [] Pivot+
    return (pole * (-longitudes / 2)).exp() >> ((across * (-colatitudes / 2)).exp() >> pole)


# One vortex of unit circulation at the north pole, and places at the angles from it along a meridian.
pole = pivot(np.array([0.0, 0.0, 1.0]))                                              # [] Pivot+
places = turned(angles, np.zeros_like(angles))                                       # [places] Pivot+
# The speed of the flow at each place, and the circulation around the circle about the vortex through it.
speeds = places.commutator(pole / (2 * np.pi * (places & pole).norm_squared())).norm()   # [places] Scalar+
circulations = speeds * (2 * np.pi * np.sin(angles))                                 # [places] Scalar+

In [ ]:
print("circulation around circles about the vortex:", circulations.to_array())

## 2. Two steps made of rotors

Two vortices alone keep the angle between them, so the sum of their two weighted bivectors stays the same while they move, and they turn rigidly about it. The explicit step turns the vortices pair by pair, each pair by its exact rotor, through the rounds of a round-robin schedule for half the step and back for the other half. The implicit step turns every vortex by the rotor of the flow of the others at the middle of its step, found by a few sweeps of fixed-point iteration: each vortex moves along an arc of a circle.

The flow keeps the vortices' impulse, the circulation-weighted sum of their bivectors, for its symmetry under the turns of the sphere. The pairs' rotors move each pair rigidly, which keeps it exactly; the midpoint turns, each vortex by its own rotor, keep it only closely. Both steps are of second order, and on two pairs leapfrogging they part by a small fraction of how far the vortices travel. A step of a handful of vortices is a few hundred small operations, the rotor exponentials above all; `jax.lax.scan` compiles a whole run of steps into one loop.

In [ ]:
# Two leapfrogging pairs on the first meridian, each vortex an angle from the equator, positive ones to the
# north, and their core; the step, the fixed-point sweeps, and the steps compared.
frog_angles = np.array([0.5, -0.5, 0.275, -0.275])
frog_circulations = np.array([1.0, -1.0, 1.0, -1.0])
pair_core = 0.05
dt, n_sweeps, n_compared_steps = 0.05, 3, 100

In [ ]:
def midpoint(vortices: Vortices, dt: float, n_sweeps: int) -> Vortices:
    """One step turning every vortex by the rotor of the flow of the others at the middle of its step."""
    later = vortices.pivots                                                            # [..., vortices] Pivot+
    for _ in range(n_sweeps):
        middle = replace(vortices, pivots=((vortices.pivots + later) / 2).normalized())
        later = (flow(middle, middle.pivots) * (-dt / 2)).exp() >> vortices.pivots
    return replace(vortices, pivots=later)


def paired(vortices: Vortices, partners: np.ndarray, dt: float) -> Vortices:
    """One step by the exact rotors of pairs, through the rounds [rounds, vortices] of partners and back."""
    pivots = vortices.pivots                                                           # [..., vortices] Pivot+
    for partner in np.concatenate([partners, partners[::-1]]):
        others = pivots[..., partner]                                                  # [..., vortices] Pivot+
        rate = 1 / (2 * np.pi * ((pivots.normalized() & others.normalized()).norm_squared() + vortices.core_radius**2))
        # The two bivectors weighted by circulation: the point the pair turns about.
        both = (pivots * vortices.circulations + others * vortices.circulations[partner]) * rate
        pivots = (both * (-dt / 4)).exp() >> pivots                                    # [..., vortices] Pivot+
    return replace(vortices, pivots=pivots)


def impulse(vortices: Vortices) -> Pivot:
    """The circulation-weighted sum of the vortices' bivectors."""
    return (vortices.pivots.normalized() * vortices.circulations).sum(axis=-1)         # [...] Pivot+


def history(vortices: Vortices, step, n_steps: int) -> Pivot:
    """The vortices' pivots at every step [steps + 1, vortices], the steps compiled into one loop."""
    def advance(pivots: Pivot, _) -> tuple[Pivot, Pivot]:
        later = step(replace(vortices, pivots=pivots)).pivots
        return later, later

    # The pivots carried into JAX for the loop, and the history back.
    start = vortices.pivots.materialize(JaxContext(ga))                                # [vortices] Pivot+
    _, later = jax.lax.scan(advance, start, length=n_steps)                            # [steps, vortices] Pivot+
    return concatenate([start[None], later]).materialize(mv.context)


frogs = Vortices(turned(np.pi / 2 - frog_angles, np.zeros(len(frog_angles))), frog_circulations, pair_core)
by_pairs = history(frogs, lambda current: paired(current, schedule(len(frog_angles)), dt), n_compared_steps)
by_midpoints = history(frogs, lambda current: midpoint(current, dt, n_sweeps), n_compared_steps)
impulse_changes = {name: impulse(replace(frogs, pivots=pivots)) - impulse(frogs)
                   for name, pivots in (("pairs", by_pairs), ("midpoints", by_midpoints))}
parting = (by_pairs[-1] - by_midpoints[-1]).norm_squared()                             # [vortices] Scalar+
travel = (by_midpoints[-1] - by_midpoints[0]).norm_squared()                           # [vortices] Scalar+

In [ ]:
for name, change in impulse_changes.items():
    print(f"largest change of the impulse by {name}: {change.norm().to_array().max():.1e}")
print(f"the steps part by {np.sqrt(parting.to_array().max()):.3f}; the vortices travel {np.sqrt(travel.to_array().max()):.2f}")

## 3. Rings

A ring of equal vortices around the point facing the viewer turns rigidly about that point. Every vortex starts a random ten-thousandth off its place. In the plane, a ring of up to six such vortices is stable and a ring of seven is on the edge; on the sphere, at a quarter of a right angle from the pole, the rings of five and six turn on undisturbed and the ring of seven breaks up, one vortex falling into the middle. Each vortex is drawn at both its points, on whichever side faces the viewer.

In [ ]:
# The rings' sizes, their angle from the pole and the vortices' core; the disturbance and its seed; the steps of the
# rings and of the travelling pairs and the steps between their frames, the view as elevation and azimuth in
# degrees, and the duration of a frame.
ring_sizes, ring_angle, ring_core = (5, 6, 7), np.radians(25.0), 0.02
disturbance, seed = 1e-4, 0
ring_steps, ring_steps_per_frame = 320, 2
travel_steps, travel_steps_per_frame = 640, 4
ring_view, duration_ms = (70.0, -60.0), 80

In [ ]:
rng = np.random.default_rng(seed)


def ring(size: int) -> Vortices:
    """A ring of equal vortices of the given size around the pole, each place nudged a little at random."""
    places = turned(np.full(size, ring_angle), 2 * np.pi * np.arange(size) / size)  # [vortices] Pivot+
    nudges = (pivot(rng.normal(size=(size, 3))) * disturbance).exp()                # [vortices] Rotor+
    return Vortices(nudges >> places, np.ones(size), ring_core)


rings = [history(ring(size), lambda current: midpoint(current, dt, n_sweeps), ring_steps) for size in ring_sizes]
ring_chords = [(pivots[:, :, None] - pivots[:, None, :]).norm_squared() for pivots in rings]   # [steps + 1, vortices, vortices] Scalar+
show_animation(render.animate(rings, ring_view, ring_steps_per_frame), "sphere_vortices_rings", duration_ms)

## 4. Travelling around the sphere

A pair of opposite vortices, side by side across the equator, travels along it, around the whole sphere; two such pairs, the inner narrower, leapfrog as they go, taking turns passing through each other, as in the plane. The positive vortices stay mirror images of the negative ones across the equator. This cell takes about thirty seconds, most of it drawing the frames.

In [ ]:
# A dipole on the first meridian, each vortex an angle from the equator, and the view.
dipole_angles = np.array([0.3, -0.3])
dipole_circulations = np.array([1.0, -1.0])
travel_view = (20.0, 0.0)

In [ ]:
dipole = Vortices(turned(np.pi / 2 - dipole_angles, np.zeros(2)), dipole_circulations, pair_core)
travelling = [history(vortices, lambda current: midpoint(current, dt, n_sweeps), travel_steps) for vortices in (dipole, frogs)]
# The mirror images in the equator of the positive vortices' flows, circulation times bivector.
mirrored = mv.z >> (travelling[1][:, ::2] * frogs.circulations[::2])               # [steps + 1, pairs] Pivot+
show_animation(render.animate(travelling, travel_view, travel_steps_per_frame), "sphere_vortices_travelling", duration_ms)

In [ ]:
# checks
# The flow is irrotational away from the vortex's points.
np.testing.assert_allclose(circulations.to_array(), 1.0, rtol=1e-12)
# The pairs keep the impulse to round-off, the midpoint turns closely; the steps part by a small fraction
# of the travel.
np.testing.assert_allclose(impulse_changes["pairs"].kernel, 0.0, atol=1e-12)
np.testing.assert_allclose(impulse_changes["midpoints"].kernel, 0.0, atol=1e-2)
assert np.sqrt(parting.to_array().max()) < 0.05 * np.sqrt(travel.to_array().max())
# The rings of five and six keep their shape; the ring of seven breaks up.
changes = [np.abs(chords.to_array() - chords.to_array()[0]).max() for chords in ring_chords]
assert changes[0] < 0.01 and changes[1] < 0.05 and changes[2] > 0.2
# The leapfrogging pairs stay mirror images across the equator: the mirror takes each positive vortex's flow
# onto the flow of the negative vortex beside it.
np.testing.assert_allclose((mirrored - travelling[1][:, 1::2] * frogs.circulations[1::2]).kernel, 0.0, atol=1e-8)